# 🎓 AuraTutor - Lớp Học Trực Tuyến 1:1 Thời Gian Thực (Google Colab GPU)

Notebook này cho phép bạn chạy toàn bộ hệ thống gia sư AI tương tác thời gian thực (**Wav2Lip + LiveTalking + WebRTC + EdgeTTS**) trên **GPU Tesla T4 / A100 miễn phí của Google Colab**.

### ⚡ Ưu điểm vượt trội so với chạy trên máy cục bộ:
* **Tốc độ suy luận đạt 60 – 80+ FPS** (gấp 3-4 lần so với card GTX 1050 Ti).
* **16GB VRAM** chuẩn máy chủ, hoàn toàn không nóng máy tính cá nhân, tiết kiệm pin.
* Đường truyền mạng máy chủ Google 1 Gbps giúp gọi WebRTC và Edge-TTS gần như không có độ trễ.
* Có link công khai HTTPS truy cập được từ mọi thiết bị (Laptop, Điện thoại, Máy tính bảng).

---

### Bước 1: Kiểm tra phần cứng GPU (Bắt buộc chọn GPU: Runtime -> Change runtime type -> T4 GPU)

In [ ]:
!nvidia-smi

### Bước 2: Tải mã nguồn AuraTutor và cài đặt thư viện cần thiết

In [ ]:
!git clone https://github.com/TontonYuta/AuraTutor-LiveTalking.git
%cd AuraTutor-LiveTalking
!pip install -q -r requirements.txt
!pip install -q edge-tts aiohttp-cors openai python-dotenv

### Bước 3: Tải Model Weights Wav2Lip và Dữ liệu Avatar từ Hugging Face

In [ ]:
# Tải mô hình Wav2Lip (214 MB) và avatar thầy giáo kính trắng
!mkdir -p models data/avatars
!curl -L -o models/wav2lip.pth https://huggingface.co/shibing624/ai-avatar-wav2lip/resolve/main/wav2lip.pth
!curl -L -o avatar.tar.gz https://huggingface.co/shibing624/ai-avatar-wav2lip/resolve/main/wav2lip_avatar_glass_man.tar.gz
!tar -xzf avatar.tar.gz -C data/avatars/
!rm -f avatar.tar.gz
print("✅ Tải weights và avatar hoàn tất!")

### Bước 4: Thiết lập Cloudflare Tunnel (Tạo link truy cập công khai HTTPS)

In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
import subprocess, time, re

# Khởi động tunnel ngầm
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8010"], stderr=subprocess.PIPE, stdout=subprocess.PIPE, text=True)
time.sleep(3)

# Đọc URL công khai từ cloudflared log
public_url = None
for line in tunnel_proc.stderr:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print("=" * 60)
print("🌐 LINK VÀO LỚP HỌC TRỰC TUYẾN CỦA BẠN:")
print(f"{public_url}/index.html")
print("=" * 60)

### Bước 5: Khởi động Máy chủ Lớp học AI (Giọng nói Tiếng Việt NamMinhNeural)

In [ ]:
!python app.py \
    --transport webrtc \
    --model wav2lip \
    --avatar_id wav2lip_avatar_glass_man \
    --tts edgetts \
    --REF_FILE vi-VN-NamMinhNeural \
    --listenport 8010